In [1]:
import os, time, json, uuid, boto3, logging, requests
from datetime import datetime

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name

bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime")

timestamp_str = time.strftime("%Y%m%d%H%M%S")[-7:]
suffix = f"{timestamp_str}"


In [2]:
url = "https://raw.githubusercontent.com/aws-samples/amazon-bedrock-samples/main/rag/knowledge-bases/features-examples/utils/knowledge_base.py"
os.makedirs("utils", exist_ok=True)

response = requests.get(url)
with open("utils/knowledge_base.py", "w") as f:
    f.write(response.text)

from utils.knowledge_base import BedrockKnowledgeBase


In [3]:
knowledge_base_name = f"hr-agent-knowledge-base-{suffix}"
knowledge_base_description = "HR onboarding and benefits KB"

data_bucket_name = f"bedrock-hr-agent-{suffix}-bucket"
data_sources = [{"type": "S3", "bucket_name": data_bucket_name}]


In [4]:
import botocore

def create_s3_bucket(bucket_name, region):
    s3 = boto3.client('s3', region_name=region)
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )

create_s3_bucket(data_bucket_name, region)


In [5]:
def upload_directory(path, bucket):
    for root, _, files in os.walk(path):
        for file in files:
            s3_client.upload_file(
                os.path.join(root, file),
                bucket,
                file
            )

upload_directory("./onboarding_files", data_bucket_name)


In [6]:
knowledge_base = BedrockKnowledgeBase(
    kb_name=knowledge_base_name,
    kb_description=knowledge_base_description,
    data_sources=data_sources,
    chunking_strategy="FIXED_SIZE",
    suffix=f"{suffix}-f"
)


Step 1 - Creating or retrieving S3 bucket(s) for Knowledge Base documents
['bedrock-hr-agent-6093124-bucket']
buckets_to_check:  ['bedrock-hr-agent-6093124-bucket']
Bucket bedrock-hr-agent-6093124-bucket already exists - retrieving it!
Step 2 - Creating Knowledge Base Execution Role (AmazonBedrockExecutionRoleForKnowledgeBase_6093124-f) and Policies
Step 3a - Creating OSS encryption, network and data access policies
Step 3b - Creating OSS Collection (this step takes a couple of minutes to complete)
{ 'ResponseMetadata': { 'HTTPHeaders': { 'connection': 'keep-alive',
                                         'content-length': '320',
                                         'content-type': 'application/x-amz-json-1.0',
                                         'date': 'Fri, 26 Dec 2025 04:01:49 '
                                                 'GMT',
                                         'x-amzn-requestid': '9098a591-51c5-4374-a2c5-8852df8463d7'},
                        'HTTPStatusCod

In [7]:
time.sleep(30)
knowledge_base.start_ingestion_job()

kb_id = knowledge_base.get_knowledge_base_id()
kb_id


job 1 started successfully

{ 'dataSourceId': 'ORI7GQQ4U9',
  'ingestionJobId': 'JDYTT9XGZU',
  'knowledgeBaseId': 'MU8T54LFJV',
  'startedAt': datetime.datetime(2025, 12, 26, 4, 5, 4, 56279, tzinfo=tzutc()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 3,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 3},
  'status': 'COMPLETE',
  'updatedAt': datetime.datetime(2025, 12, 26, 4, 5, 10, 138088, tzinfo=tzutc())}
'MU8T54LFJV'............................


'MU8T54LFJV'

In [8]:
query = "Who is the medical insurance provider name?"

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={"text": query},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": kb_id,
            "modelArn": f"arn:aws:bedrock:{region}::foundation-model/amazon.nova-micro-v1:0",
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {"numberOfResults": 5}
            }
        }
    }
)

print(response["output"]["text"])


The model cannot find sufficient information to answer the question about the medical insurance provider name in the provided search results. The passages mainly discuss working hours, leave policies, code of conduct, and additional benefits like the employee assistance program (EAP), but do not specify the medical insurance provider.


In [9]:
%store kb_id
%store region


Stored 'kb_id' (str)
Stored 'region' (str)


In [10]:
%store -r kb_id
%store -r region

os.environ["KNOWLEDGE_BASE_ID"] = kb_id
os.environ["AWS_REGION"] = region
os.environ["MIN_SCORE"] = "0.2"


In [11]:
import getpass, os
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:")


In [12]:
%store -r kb_id
%store -r kb_region

import os

os.environ["KNOWLEDGE_BASE_ID"] = kb_id
os.environ["AWS_REGION"] = kb_region
os.environ["MIN_SCORE"] = "0.2"


no stored variable or alias kb_region


NameError: name 'kb_region' is not defined